In [8]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

# -------------------------
# Variables demográficas
# -------------------------

age = np.random.randint(17, 22, size=n)

father_education = np.random.choice(
    ['sin_estudios', 'primaria', 'secundaria', 'bachillerato', 'universidad', 'posgrado'],
    size=n,
    p=[0.05, 0.15, 0.25, 0.25, 0.20, 0.10]
)

mother_education = np.random.choice(
    ['sin_estudios', 'primaria', 'secundaria', 'bachillerato', 'universidad', 'posgrado'],
    size=n,
    p=[0.03, 0.12, 0.25, 0.27, 0.22, 0.11]
)

school_type = np.random.choice(
    ['publico', 'privado'],
    size=n,
    p=[0.65, 0.35]
)

scholarship = np.random.choice(
    [0, 1],
    size=n,
    p=[0.7, 0.3]
)

# -------------------------
# Hábitos y contexto
# -------------------------

smoking_frequency = np.random.choice(
    ['no_fuma', 'fuma_poco', 'fumador_ocasional', 'fumador_constante'],
    size=n,
    p=[0.55, 0.20, 0.15, 0.10]
)

study_hours_extra = np.random.choice(
    ['<5', '<10', '<15', '15+'],
    size=n,
    p=[0.25, 0.30, 0.25, 0.20]
)

extracurricular = np.random.choice(
    ['deportiva', 'cultural', 'social'],
    size=n,
    p=[0.40, 0.35, 0.25]
)

work_hours = np.clip(
    np.random.normal(12, 10, size=n),
    0,
    40
)

# -------------------------
# Desempeño académico
# -------------------------

math_grade = np.clip(np.random.normal(75, 10, size=n), 50, 100)
language_grade = np.clip(np.random.normal(78, 9, size=n), 50, 100)

# -------------------------
# Mapeos ordinales
# -------------------------

edu_map = {
    'sin_estudios': 0,
    'primaria': 1,
    'secundaria': 2,
    'bachillerato': 3,
    'universidad': 4,
    'posgrado': 5
}

smoke_map = {
    'no_fuma': 0,
    'fuma_poco': -5,
    'fumador_ocasional': -12,
    'fumador_constante': -20
}

study_map = {
    '<5': 0,
    '<10': 5,
    '<15': 10,
    '15+': 20
}

extra_map = {
    'deportiva': 15,
    'cultural': 12,
    'social': 5
}

# -------------------------
# Construcción del puntaje
# -------------------------

base_score = 900

aptitude_score = (
    900
    + 8 * pd.Series(father_education).map(edu_map)
    + 10 * pd.Series(mother_education).map(edu_map)
    + 18 * scholarship
    - pd.Series(smoking_frequency).map(smoke_map)
    + pd.Series(study_hours_extra).map(study_map)
    + pd.Series(extracurricular).map(extra_map)
    + 2.5 * math_grade
    + 2.0 * language_grade
    - 1.5 * work_hours
    + np.random.normal(0, 50, size=n)
)


aptitude_score = np.clip(aptitude_score, 700, 1300)

# -------------------------
# Dataset final
# -------------------------

df = pd.DataFrame({
    'age': age,
    'father_education': father_education,
    'mother_education': mother_education,
    'school_type': school_type,
    'scholarship': scholarship,
    'smoking_frequency': smoking_frequency,
    'study_hours_extra': study_hours_extra,
    'extracurricular': extracurricular,
    'work_hours': work_hours.round(1),
    'math_grade': math_grade.round(1),
    'language_grade': language_grade.round(1),
    'aptitude_score': aptitude_score.round(0)
})

df.head()


,age,father_education,mother_education,school_type,scholarship,smoking_frequency,study_hours_extra,extracurricular,work_hours,math_grade,language_grade,aptitude_score
0,20,posgrado,universidad,publico,0,no_fuma,<5,social,4.3,66.2,73.5,1300.0
1,21,bachillerato,primaria,publico,0,fumador_ocasional,15+,deportiva,3.3,81.6,64.3,1300.0
2,19,bachillerato,primaria,privado,0,no_fuma,15+,deportiva,19.2,71.1,86.3,1276.0
3,21,posgrado,sin_estudios,publico,0,fuma_poco,<5,cultural,16.5,54.3,74.5,1194.0
4,21,universidad,posgrado,privado,0,no_fuma,<10,deportiva,10.4,86.7,79.6,1300.0


In [16]:
df.select_dtypes(exclude='object').corr().round(2)['aptitude_score']

age              -0.01
scholarship       0.14
work_hours       -0.20
math_grade        0.32
language_grade    0.23
aptitude_score    1.00
Name: aptitude_score, dtype: float64

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   int32  
 1   father_education   1000 non-null   object 
 2   mother_education   1000 non-null   object 
 3   school_type        1000 non-null   object 
 4   scholarship        1000 non-null   int64  
 5   smoking_frequency  1000 non-null   object 
 6   study_hours_extra  1000 non-null   object 
 7   extracurricular    1000 non-null   object 
 8   work_hours         1000 non-null   float64
 9   math_grade         1000 non-null   float64
 10  language_grade     1000 non-null   float64
 11  aptitude_score     1000 non-null   float64
dtypes: float64(4), int32(1), int64(1), object(6)
memory usage: 90.0+ KB


In [17]:
df.groupby('scholarship')['aptitude_score'].mean().reset_index(name='avg').sort_values('avg')

,scholarship,avg
0,0,1273.660248
1,1,1284.659341


In [19]:
df.to_csv('csnat_2025_west.csv',index=False)